In [ ]:
# ==============================================================================
# CELL 1 / PHASE 1: MEDICAL SUBJECT HEADINGS (MeSH) INGESTION
# 
# ARCHITECTURE NOTE: MeSH uses a "Bucket" data model, not a standard W3C tree.
# - D-Nodes (Descriptors): The structural hierarchy (The Tree).
# - M-Nodes (Concepts): The ideas sitting inside the Descriptor buckets.
# - T-Nodes (Terms): The actual text strings/synonyms attached to Concepts.
#
# SSSOM ALIGNMENT STRATEGY: 
# To prevent false-equivalence in SSSOM mapping (e.g., mapping "Christianity" 
# to a node that includes "Stigmata" as an exact match), this script "flattens" 
# the MeSH buckets. It extracts the Descriptor as a parent row, and explicitly 
# pulls out 'Narrower Concepts' (M-nodes) as individual child rows. 
# For full theoretical documentation, see METHODOLOGY.md.
# ==============================================================================

import os
import sys
import requests
import pandas as pd
import time
from collections import deque
from dotenv import load_dotenv

# --- 1. Seeds & Target Scope ---
# Define the top-level URIs to extract. The script will dynamically calculate
# their base hierarchy paths before scraping.
SEED_URIS = [
    "http://id.nlm.nih.gov/mesh/D012067",      # Religion
    "http://id.nlm.nih.gov/mesh/D026443",      # Spiritual Therapies
    "http://id.nlm.nih.gov/mesh/D035621",      # Secularism
    "http://id.nlm.nih.gov/mesh/D065834",      # Religious Personnel
    "http://id.nlm.nih.gov/mesh/D064698",      # Parish Nursing
    "http://id.nlm.nih.gov/mesh/D066189",      # Missionaries
    "http://id.nlm.nih.gov/mesh/D000095083",   # Traditional Medicine Practitioners
    "http://id.nlm.nih.gov/mesh/D013483",      # Superstitions
    "http://id.nlm.nih.gov/mesh/D008493"       # Medical Missions
]

# --- 2. Environment & Setup ---
load_dotenv("../../config/.env")
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    print("CRITICAL: ingest_schema_manager.py not found. Check PYTHONPATH.")
    sys.exit(1)

SOURCE_PREFIX = "MeSH"
registry_data = get_registry_info(SOURCE_PREFIX, config_dir)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}_api.csv")

SPARQL_ENDPOINT = "https://id.nlm.nih.gov/mesh/sparql"
HEADERS = {'User-Agent': f'ReligiousMappingProject/1.0 ({contact_email})', 'Accept': 'application/sparql-results+json'}

# --- 3. Optimized Atomic Queries ---

def run_query(query, timeout=25):
    """Executes SPARQL queries with a focus on speed and reliability."""
    try:
        r = requests.get(SPARQL_ENDPOINT, params={'query': query, 'format': 'JSON'}, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        return r.json().get('results', {}).get('bindings', [])
    except Exception as e:
        print(f"\n   [!] Query Error: {str(e)[:100]}")
        return []

def get_seed_path(uri):
    """Dynamically climbs the MeSH tree to calculate the absolute breadcrumb path of a seed."""
    query = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    SELECT DISTINCT ?ancestorLabel ?ancestorTree
    WHERE {{
      {{ SELECT ?tree WHERE {{ <{uri}> meshv:treeNumber ?tree }} LIMIT 1 }}
      ?tree meshv:parentTreeNumber* ?ancestorTree .
      ?ancestor meshv:treeNumber ?ancestorTree .
      ?ancestor rdfs:label ?ancestorLabel .
    }}
    """
    res = run_query(query, timeout=15)
    if not res: return ""
    
    parsed_results = []
    for r in res:
        parsed_results.append((len(r['ancestorTree']['value']), r['ancestorLabel']['value']))
        
    parsed_results = sorted(list(set(parsed_results)), key=lambda x: x[0])
    if parsed_results:
        # Exclude the very last item, as it is the seed itself
        return " > ".join([p[1] for p in parsed_results[:-1]])
    return ""

def get_node_metadata(uri):
    """Fetches D-Node data. Offloads string joining to Python for server stability."""
    query = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    SELECT ?label ?scopeNote ?active ?pID ?alt
    WHERE {{
      <{uri}> rdfs:label ?label .
      OPTIONAL {{ <{uri}> meshv:active ?active . }}
      OPTIONAL {{ <{uri}> meshv:preferredConcept ?pref . 
                  OPTIONAL {{ ?pref meshv:scopeNote ?scopeNote . }}
                  OPTIONAL {{ ?pref (meshv:preferredTerm|meshv:term)/meshv:prefLabel ?alt . }}
               }}
      OPTIONAL {{ <{uri}> meshv:treeNumber/meshv:parentTreeNumber/^meshv:treeNumber ?pDesc . 
                  BIND(REPLACE(STR(?pDesc), "http://id.nlm.nih.gov/mesh/", "") AS ?pID) }}
    }}"""
    res = run_query(query)
    if not res: return None
    
    label = res[0]['label']['value']
    return {
        'label': label,
        'scopeNote': res[0].get('scopeNote', {}).get('value', ''),
        'status': 'active' if res[0].get('active', {}).get('value') == 'true' else 'inactive',
        'parents': " | ".join(sorted(list(set([r['pID']['value'] for r in res if 'pID' in r])))),
        'syns': " | ".join(sorted(list(set([r['alt']['value'] for r in res if 'alt' in r and r['alt']['value'] != label]))))
    }

def get_m_node_uris(uri):
    """Identifies ALL Concepts (M-Nodes) within a Descriptor bucket, excluding only the Preferred Concept."""
    query = f"""
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    SELECT DISTINCT ?m WHERE {{ 
        <{uri}> meshv:concept ?m . 
        <{uri}> meshv:preferredConcept ?p . 
        FILTER(?m != ?p)
    }}"""
    return [b['m']['value'] for b in run_query(query)]

def get_m_metadata(uri):
    """Fetches M-Node data with exact-match synonyms."""
    query = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    SELECT ?label ?scopeNote ?active ?alt
    WHERE {{ 
        <{uri}> rdfs:label ?label . 
        OPTIONAL {{ <{uri}> meshv:active ?active . }} 
        OPTIONAL {{ <{uri}> meshv:scopeNote ?scopeNote . }} 
        OPTIONAL {{ <{uri}> (meshv:preferredTerm|meshv:term)/meshv:prefLabel ?alt . }}
    }}"""
    res = run_query(query)
    if not res: return None
    
    label = res[0]['label']['value']
    return {
        'label': label,
        'scopeNote': res[0].get('scopeNote', {}).get('value', ''),
        'status': 'active' if res[0].get('active', {}).get('value') == 'true' else 'inactive',
        'syns': " | ".join(sorted(list(set([r['alt']['value'] for r in res if 'alt' in r and r['alt']['value'] != label]))))
    }

def get_children(uri):
    """Identifies child D-Nodes on the primary MeSH tree."""
    query = f"""
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    SELECT DISTINCT ?c WHERE {{ 
        <{uri}> meshv:treeNumber ?pt . ?ct meshv:parentTreeNumber ?pt . ?c meshv:treeNumber ?ct .
    }}"""
    return [b['c']['value'] for b in run_query(query)]

# --- 4. Iterative Queue Processor ---

def scrape_mesh(seed_uri, base_path, visited):
    """Iteratively crawls a MeSH branch starting from a seed URI."""
    queue = deque([(seed_uri, base_path, 0)])
    
    while queue:
        uri, path, depth = queue.popleft()
        if uri in visited: continue
        
        cid = uri.split('/')[-1]
        print(f"\r  [Depth {depth}] Processing {cid}...", end="", flush=True)
        
        # 1. Process D-Node
        data = get_node_metadata(uri)
        if not data:
            visited.add(uri)
            continue
            
        label = data['label']
        # If the base_path is completely empty, don't prepend a dangling separator
        current_path = f"{path} > {label}" if path else label
        
        row = finalize_row({
            "Source_System": SOURCE_NAME, "Primary_Label": label, 
            "CURIE": f"MeSH:{cid}", "Concept_Type": "meshv:TopicalDescriptor", 
            "Hierarchy_Path": current_path, "Synonyms": data['syns'], 
            "Description": data['scopeNote'], "Parent_IDs": data['parents'],
            "Concept_ID": cid, "URI": uri, "Status": data['status']
        })
        pd.DataFrame([row])[COLUMN_ORDER].to_csv(
            raw_ingest_file, mode='a', index=False, 
            header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
        )
        visited.add(uri)

        # 2. Process internal M-Nodes
        m_uris = get_m_node_uris(uri)
        for m_uri in m_uris:
            if m_uri in visited: continue
            m_data = get_m_metadata(m_uri)
            if m_data:
                m_label = m_data['label']
                m_cid = m_uri.split('/')[-1]
                m_row = finalize_row({
                    "Source_System": SOURCE_NAME, "Primary_Label": m_label, 
                    "CURIE": f"MeSH:{m_cid}", "Concept_Type": "meshv:Concept", 
                    "Hierarchy_Path": f"{current_path} > {m_label}", "Synonyms": m_data['syns'], 
                    "Description": m_data['scopeNote'], "Parent_IDs": cid,
                    "Concept_ID": m_cid, "URI": m_uri, "Status": m_data['status']
                })
                pd.DataFrame([m_row])[COLUMN_ORDER].to_csv(raw_ingest_file, mode='a', index=False, header=False, encoding='utf-8-sig')
                visited.add(m_uri)

        # 3. Queue Tree Children
        child_uris = get_children(uri)
        for c in child_uris:
            if c not in visited:
                queue.append((c, current_path, depth + 1))
        
        time.sleep(0.3) # Politeness delay

# --- 5. Main Execution ---

if __name__ == "__main__":
    if os.path.exists(raw_ingest_file):
        os.remove(raw_ingest_file)

    global_visited = set()
    start_time = time.time()

    print(f"[*] Starting Full MeSH Domain Ingest...\n")

    for seed in SEED_URIS:
        print(f"[{seed.split('/')[-1]}] Calculating dynamic base path...")
        dynamic_path = get_seed_path(seed)
        print(f"[*] Seed Path resolved: {dynamic_path}")
        
        scrape_mesh(seed, dynamic_path, global_visited)
        print("\n")

    elapsed = (time.time() - start_time) / 60
    print(f"\n[SUCCESS] Extraction complete in {elapsed:.2f} minutes.")
    print(f"[*] Total distinct concepts captured: {len(global_visited)}")

In [ ]:
# ==============================================================================
# CELL 2 / PHASE 2: MeSH LATERAL DISCOVERY (SPARQL-Live Inbox Approach)
# Run this cell to harvest lateral links outside the current ingested scope.
# Tags candidates with a 'Discovery_Pass' to keep new items separate from old.
# ==============================================================================

import os
import pandas as pd
import requests
import time
from dotenv import load_dotenv

RUN_LATERAL_DISCOVERY = True # Toggle to False once ingest script is finalized

if RUN_LATERAL_DISCOVERY:
    # --- 1. Setup & Baseline Load ---
    load_dotenv("../../config/.env")
    contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")
    current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
    raw_data_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "raw"))
    
    raw_mesh_file = os.path.join(raw_data_dir, "raw_mesh_api.csv")
    output_candidates_file = os.path.join(raw_data_dir, "mesh_lateral_candidates.csv")
    
    SPARQL_ENDPOINT = "https://id.nlm.nih.gov/mesh/sparql"
    HEADERS = {'User-Agent': f'ReligiousMappingProject/1.0 ({contact_email})', 'Accept': 'application/sparql-results+json'}
    
    if not os.path.exists(raw_mesh_file):
        raise FileNotFoundError(f"Please run Phase 1 to generate the baseline {raw_mesh_file} first.")
        
    existing_df = pd.read_csv(raw_mesh_file)
    existing_uris = set(existing_df['URI'].astype(str).str.strip())
    print(f"[*] Loaded {len(existing_uris)} ingested MeSH concepts as the baseline.")

    # --- 2. Build Suppression List & Determine Pass Number ---
    seen_uris = set(existing_uris)
    old_candidates_df = pd.DataFrame()
    current_pass = 1
    
    if os.path.exists(output_candidates_file):
        old_candidates_df = pd.read_csv(output_candidates_file)
        if 'Discovery_Pass' in old_candidates_df.columns:
            current_pass = old_candidates_df['Discovery_Pass'].max() + 1
        else:
            old_candidates_df['Discovery_Pass'] = 1
            current_pass = 2
            
        if 'Candidate_URI' in old_candidates_df.columns:
            for uri in old_candidates_df['Candidate_URI']:
                seen_uris.add(str(uri).strip())
                
    print(f"[*] Built suppression list of {len(seen_uris)} total URIs (Ingested + Previously Reviewed).")
    print(f"[*] Tagging new discoveries as Pass {current_pass}.")

    # --- 3. Helper Functions ---
    def get_lateral_links(uri):
        """Finds descriptors linked via seeAlso or preferredConcept/relatedConcept."""
        query = f"""
        PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        
        SELECT DISTINCT ?related ?label
        WHERE {{
          {{ <{uri}> meshv:seeAlso ?related . }}
          UNION
          {{ <{uri}> meshv:preferredConcept/meshv:relatedConcept/^meshv:concept ?related . }}
          
          ?related rdfs:label ?label .
        }}"""
        try:
            r = requests.get(SPARQL_ENDPOINT, params={'query': query, 'format': 'JSON'}, headers=HEADERS, timeout=15)
            return r.json().get('results', {}).get('bindings', [])
        except:
            return []

    def get_hierarchy_path(uri):
        """Climbs the MeSH tree to calculate the absolute breadcrumb path of the candidate."""
        query = f"""
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
        SELECT DISTINCT ?ancestorLabel ?ancestorTree
        WHERE {{
          {{ SELECT ?tree WHERE {{ <{uri}> meshv:treeNumber ?tree }} LIMIT 1 }}
          ?tree meshv:parentTreeNumber* ?ancestorTree .
          ?ancestor meshv:treeNumber ?ancestorTree .
          ?ancestor rdfs:label ?ancestorLabel .
        }}
        """
        try:
            r = requests.get(SPARQL_ENDPOINT, params={'query': query, 'format': 'JSON'}, headers=HEADERS, timeout=15)
            results = r.json().get('results', {}).get('bindings', [])
            
            parsed_results = []
            for res in results:
                tree_val = res['ancestorTree']['value']
                label_val = res['ancestorLabel']['value']
                parsed_results.append((len(tree_val), label_val))
                
            parsed_results = sorted(list(set(parsed_results)), key=lambda x: x[0])
            
            if parsed_results:
                return " > ".join([p[1] for p in parsed_results])
        except Exception:
            pass
        return "Unknown Path"

    # --- 4. Discovery Traversal ---
    candidates_dict = {}
    # We only scan TopicalDescriptors for lateral links to keep it focused
    baseline_descriptors = existing_df[existing_df['Concept_Type'] == 'meshv:TopicalDescriptor']['URI'].tolist()
    
    print(f"\nScanning live MeSH SPARQL endpoint for lateral associations...")
    
    for idx, uri_str in enumerate(baseline_descriptors, 1):
        print(f"\rScanning baseline descriptor {idx}/{len(baseline_descriptors)}: {uri_str.split('/')[-1]}...", end="", flush=True)
        
        related_data = get_lateral_links(uri_str)
        
        for item in related_data:
            c_uri = item['related']['value']
            
            # Check against the master suppression list (only pull Descriptors)
            if "/mesh/D" in c_uri and c_uri not in seen_uris:
                if c_uri not in candidates_dict:
                    c_label = item['label']['value']
                    
                    # Look up the human-readable path to match AFSET schema
                    c_path = get_hierarchy_path(c_uri)
                    
                    candidates_dict[c_uri] = {
                        'Discovery_Pass': current_pass,
                        'Candidate_URI': c_uri,
                        'Candidate_Label': c_label,
                        'Hierarchy_Path': c_path
                    }
        time.sleep(0.3) # Politeness

    # --- 5. Export and Tracking ---
    print(f"\n\nLateral Discovery Complete! Found {len(candidates_dict)} NET-NEW candidates.")
    
    if candidates_dict:
        new_candidates_df = pd.DataFrame(list(candidates_dict.values()))
        combined_df = pd.concat([old_candidates_df, new_candidates_df], ignore_index=True)
        
        # Sort by Discovery_Pass (Newest First), then by Hierarchy Path
        combined_df = combined_df.sort_values(by=['Discovery_Pass', 'Hierarchy_Path'], ascending=[False, True])
        
        # Ensure exact column match with AFSET
        col_order = ['Discovery_Pass', 'Candidate_URI', 'Candidate_Label', 'Hierarchy_Path']
        combined_df = combined_df[col_order]
        
        combined_df.to_csv(output_candidates_file, index=False, encoding='utf-8-sig')
        print(f"Updated {os.path.basename(output_candidates_file)} with new candidates for review.")
    else:
        print("No new relevant concepts found on this pass.")
else:
    print("Lateral Discovery is disabled.")